# Approximation Algorithms for Steiner Trees in Weighted Graphs

Nikola Labus — Naučno izračunavanje 2025/26

In [6]:
import networkx as nx
import time
from itertools import combinations
from pathlib import Path

## STP Parser

In [7]:
def parse_stp(filepath):
    G = nx.Graph()
    terminals = []
    name = ""
    section = None

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('33D32945'):
                continue

            if line.startswith('SECTION'):
                section = line.split()[1]
                continue
            if line == 'END' or line == 'EOF':
                section = None
                continue

            if section == 'Comment':
                if line.startswith('Name'):
                    name = line.split('"')[1]

            elif section == 'Graph':
                if line.startswith('Nodes'):
                    n = int(line.split()[1])
                    G.add_nodes_from(range(1, n + 1))
                elif line.startswith('E '):
                    parts = line.split()
                    u, v, w = int(parts[1]), int(parts[2]), int(parts[3])
                    G.add_edge(u, v, weight=w)

            elif section == 'Terminals':
                if line.startswith('T '):
                    terminals.append(int(line.split()[1]))

    return G, terminals, name

## Brute Force (Egzaktan algoritam)

Za svaki podskup neterminalnih čvorova proveravamo da li se terminali mogu povezati
kroz indukovani podgraf. Pamtimo minimalno razapinjuće stablo sa najmanjom težinom.

In [8]:
def brute_force_steiner(G, terminals):
    terminal_set = set(terminals)
    non_terminals = [v for v in G.nodes() if v not in terminal_set]
    best_weight = float('inf')
    best_tree = None
    subsets_checked = 0

    for k in range(len(non_terminals) + 1):
        for subset in combinations(non_terminals, k):
            subsets_checked += 1
            nodes = terminal_set | set(subset)
            subgraph = G.subgraph(nodes)

            if nx.is_connected(subgraph):
                mst = nx.minimum_spanning_tree(subgraph)
                weight = mst.size(weight='weight')
                if weight < best_weight:
                    best_weight = weight
                    best_tree = mst

    return best_tree, best_weight, subsets_checked